# AXIOM Audit S55 — Inventario y Verificación de Simetría**Para:** Abraham Fuenmayor Hernández (Altair)**Propósito:** Verificar que Drive y GitHub tienen los mismos archivos AXIOM, con los mismos SHA-256, y que el hashchain está íntegro.## Qué hace este notebook1. Lista todos los archivos de `SPEL 4.0/AXIOM/` en tu Drive2. Lista todos los archivos de `sandbox33/SPEL-4.0/.axiom/` en tu GitHub3. Compara ambas mesas y detecta divergencias4. Valida el hashchain (los bloques deben estar encadenados correctamente)5. Valida el XML contra el XSD canónico6. Te muestra un reporte final claro: ✅ todo bien o ❌ hay X problema## Qué NO hace- No te pide passphrase de minisign- No descarga ni toca tu clave privada- No modifica nada en Drive ni GitHub- Solo lee y reporta## Cómo usarlo1. Sube este archivo a tu Drive en `SPEL 4.0/AXIOM/AXIOM_AUDIT_S55.ipynb`2. Abre el archivo desde Drive (te abre Colab automáticamente)3. Menú: **Runtime → Run all**4. Te pedirá autorizar Drive (acepta) y un token GitHub (te explico abajo cómo obtenerlo)5. Lee el reporte final

## Antes de ejecutar — obtener token GitHub (3 minutos, solo la primera vez)1. En GitHub web (móvil o desktop), ve a: `Settings → Developer settings → Personal access tokens → Fine-grained tokens`2. Click **Generate new token**3. Configura:   - **Token name:** `AXIOM_AUDIT_READ_ONLY`   - **Expiration:** 90 días (renovable después)   - **Repository access:** Only select repositories → marca `sandbox33/SPEL-4.0`   - **Permissions → Repository permissions:**     - `Contents`: **Read-only**     - `Metadata`: **Read-only** (automático)     - Todo lo demás: déjalo en "No access"4. Click **Generate token**5. Copia el token que aparece (empieza con `github_pat_...`)6. **Guárdalo inmediatamente en Bitwarden** como Secure Note `GITHUB_TOKEN_AUDIT_READ_ONLY`7. Cuando este notebook lo pida, pégalo**Importante:** este token tiene permisos de SOLO LECTURA. Aunque alguien lo robe, no puede modificar ni borrar nada. Solo leer archivos del repo.

## Paso 1 — Instalar dependenciasColab no trae `lxml` preinstalado. Lo necesitamos para validar el XML contra el XSD.

In [ ]:
!pip install lxml requests --quiet
print("✅ Dependencias instaladas")

## Paso 2 — Montar Google DriveAl ejecutar esta celda, Colab te pedirá autorización para acceder a tu Drive. Es seguro: solo lee la carpeta AXIOM, no toca nada más.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_AXIOM = '/content/drive/MyDrive/ORDEN/SPEL 4.0/AXIOM'

if os.path.exists(DRIVE_AXIOM):
    print(f"✅ Drive montado, carpeta AXIOM encontrada en:")
    print(f"   {DRIVE_AXIOM}")
else:
    print(f"❌ NO encontré la carpeta AXIOM en:")
    print(f"   {DRIVE_AXIOM}")
    print(f"\nVerifica que la ruta exacta en tu Drive sea:")
    print(f"   Mi Drive/ORDEN/SPEL 4.0/AXIOM/")
    print(f"\nSi tu estructura es diferente, edita la variable DRIVE_AXIOM arriba.")

## Paso 3 — Inventario completo de DriveListamos cada archivo en `SPEL 4.0/AXIOM/` con su tamaño, fecha de modificación, y SHA-256.

In [ ]:
import os
import hashlib
from datetime import datetime

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def inventory_folder(root):
    items = []
    if not os.path.exists(root):
        return items
    for dirpath, dirnames, filenames in os.walk(root):
        for fname in filenames:
            full_path = os.path.join(dirpath, fname)
            rel_path = os.path.relpath(full_path, root)
            try:
                size = os.path.getsize(full_path)
                mtime = datetime.fromtimestamp(os.path.getmtime(full_path)).isoformat()
                sha = sha256_file(full_path)
                items.append({
                    'rel_path': rel_path,
                    'size': size,
                    'mtime': mtime,
                    'sha256': sha
                })
            except Exception as e:
                items.append({'rel_path': rel_path, 'error': str(e)})
    return items

drive_items = inventory_folder(DRIVE_AXIOM)

print(f"📁 DRIVE — {len(drive_items)} archivos encontrados en SPEL 4.0/AXIOM/\n")
print(f"{'ARCHIVO':<55} {'TAMAÑO':>10} {'SHA (primeros 12)':<14}")
print("─" * 80)
for item in sorted(drive_items, key=lambda x: x['rel_path']):
    if 'error' in item:
        print(f"  ❌ {item['rel_path']:<53} ERROR: {item['error']}")
    else:
        print(f"  {item['rel_path']:<53} {item['size']:>9}B {item['sha256'][:12]}")

## Paso 4 — Conectar con GitHubTe pediré tu token GitHub. **No se mostrará en pantalla** mientras lo pegas (eso es seguridad). Pega y dale Enter.

In [ ]:
from getpass import getpass

GITHUB_TOKEN = getpass("Pega tu GitHub token (no se mostrará): ")
GITHUB_OWNER = "sandbox33"  # Cambia si tu username es distinto
GITHUB_REPO = "SPEL-4.0"
GITHUB_PATH = ".axiom"

print(f"\n✅ Token recibido ({len(GITHUB_TOKEN)} caracteres)")
print(f"   Repo objetivo: {GITHUB_OWNER}/{GITHUB_REPO}")
print(f"   Carpeta: {GITHUB_PATH}/")

## Paso 5 — Inventario completo de GitHubUsamos la API REST de GitHub para listar archivos sin clonar el repo completo (más rápido, menos consumo).

In [ ]:
import requests
import base64

HEADERS = {
    'Authorization': f'Bearer {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github.v3+json'
}

def github_list_tree(owner, repo, path=''):
    """Lista recursivamente el contenido de una carpeta en el repo."""
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
    r = requests.get(url, headers=HEADERS)
    if r.status_code == 404:
        return []
    if r.status_code != 200:
        print(f"❌ Error API GitHub: {r.status_code} — {r.text[:200]}")
        return []
    items = []
    for entry in r.json():
        if entry['type'] == 'file':
            items.append({
                'rel_path': entry['path'].replace(f'{GITHUB_PATH}/', '', 1) if entry['path'].startswith(f'{GITHUB_PATH}/') else entry['path'],
                'size': entry['size'],
                'sha_git': entry['sha'],  # SHA de Git (no SHA-256 del contenido)
                'download_url': entry['download_url']
            })
        elif entry['type'] == 'dir':
            items.extend(github_list_tree(owner, repo, entry['path']))
    return items

def github_sha256(url):
    """Descarga un archivo y calcula su SHA-256 real (no el de Git)."""
    r = requests.get(url, headers=HEADERS)
    if r.status_code == 200:
        return hashlib.sha256(r.content).hexdigest()
    return None

print("⏳ Listando archivos de GitHub...")
github_items_raw = github_list_tree(GITHUB_OWNER, GITHUB_REPO, GITHUB_PATH)

print(f"\n📁 GITHUB — {len(github_items_raw)} archivos encontrados en {GITHUB_PATH}/\n")
print(f"{'ARCHIVO':<55} {'TAMAÑO':>10} {'SHA256 (primeros 12)':<14}")
print("─" * 80)

github_items = []
for item in sorted(github_items_raw, key=lambda x: x['rel_path']):
    sha = github_sha256(item['download_url'])
    github_items.append({**item, 'sha256': sha})
    sha_display = sha[:12] if sha else "ERROR"
    print(f"  {item['rel_path']:<53} {item['size']:>9}B {sha_display}")

## Paso 6 — Comparación de simetría Drive ↔ GitHubAquí está lo crítico. Comparamos ambos inventarios:- **Archivos solo en Drive** (faltan en GitHub) — riesgo de pérdida si Drive falla- **Archivos solo en GitHub** (faltan en Drive) — Drive desactualizado- **Archivos en ambos PERO con SHA distinto** — divergencia, requiere atenciónTu Ley de Simetría exige que Drive y GitHub coincidan exactamente.

In [ ]:
drive_map = {item['rel_path']: item for item in drive_items if 'sha256' in item}
github_map = {item['rel_path']: item for item in github_items if item.get('sha256')}

drive_paths = set(drive_map.keys())
github_paths = set(github_map.keys())

only_in_drive = drive_paths - github_paths
only_in_github = github_paths - drive_paths
in_both = drive_paths & github_paths

print("=" * 70)
print("REPORTE DE SIMETRÍA Drive ↔ GitHub")
print("=" * 70)

# Solo en Drive
if only_in_drive:
    print(f"\n⚠️  SOLO EN DRIVE ({len(only_in_drive)} archivos):")
    for p in sorted(only_in_drive):
        print(f"    {p}")
else:
    print("\n✅ No hay archivos exclusivos en Drive")

# Solo en GitHub
if only_in_github:
    print(f"\n⚠️  SOLO EN GITHUB ({len(only_in_github)} archivos):")
    for p in sorted(only_in_github):
        print(f"    {p}")
else:
    print("\n✅ No hay archivos exclusivos en GitHub")

# Divergencia de SHA
divergent = []
for p in in_both:
    if drive_map[p]['sha256'] != github_map[p]['sha256']:
        divergent.append({
            'path': p,
            'drive_sha': drive_map[p]['sha256'][:12],
            'github_sha': github_map[p]['sha256'][:12]
        })

if divergent:
    print(f"\n❌ DIVERGENCIA DE SHA ({len(divergent)} archivos con contenido distinto):")
    for d in sorted(divergent, key=lambda x: x['path']):
        print(f"    {d['path']}")
        print(f"      Drive:  {d['drive_sha']}...")
        print(f"      GitHub: {d['github_sha']}...")
else:
    print(f"\n✅ Todos los archivos en ambas mesas tienen SHA idéntico ({len(in_both)} archivos)")

# Veredicto
print("\n" + "=" * 70)
if not only_in_drive and not only_in_github and not divergent:
    print("✅✅✅ SIMETRÍA PERFECTA — Drive y GitHub están sincronizados")
else:
    print("⚠️  ATENCIÓN — Existe asimetría. Revisa el detalle arriba.")
print("=" * 70)

## Paso 7 — Validación criptográfica del hashchainEl hashchain de AXIOM debe estar íntegro: cada bloque referencia al anterior.

In [ ]:
import json

hashchain_path = os.path.join(DRIVE_AXIOM, 'axiom_hashchain.jsonl')

if not os.path.exists(hashchain_path):
    print(f"❌ No encuentro axiom_hashchain.jsonl en Drive")
else:
    with open(hashchain_path) as f:
        blocks = [json.loads(line) for line in f if line.strip()]

    print(f"📦 Hashchain encontrado con {len(blocks)} bloques\n")

    prev = None
    all_ok = True
    for b in blocks:
        seq = b.get('seq', '?')
        action = b.get('action', '?')[:50]

        if prev is None:
            # Génesis: prev_hash debe ser ceros
            if b.get('prev_hash') == '0' * 64:
                print(f"  ✅ Bloque {seq}: GÉNESIS válido — {action}")
            else:
                print(f"  ❌ Bloque {seq}: GÉNESIS con prev_hash NO-ZERO — {action}")
                all_ok = False
        else:
            # Bloque normal: prev_hash == curr_hash del anterior
            if b.get('prev_hash') == prev.get('curr_hash'):
                print(f"  ✅ Bloque {seq}: encadenado correctamente — {action}")
            else:
                print(f"  ❌ Bloque {seq}: prev_hash NO COINCIDE con bloque {prev.get('seq')} — {action}")
                all_ok = False
        prev = b

    print()
    if all_ok:
        print("✅✅✅ HASHCHAIN ÍNTEGRO — Toda la cadena verificada criptográficamente")
    else:
        print("❌ HASHCHAIN ROTO — Hay inconsistencias. NO continuar sin resolver.")

## Paso 8 — Validación del XML maestro contra el XSD canónicoEl `axiom_master.xml` debe validar estrictamente contra `axiom_schema.xsd`. Si no valida, hay un problema estructural.

In [ ]:
from lxml import etree

xml_path = os.path.join(DRIVE_AXIOM, 'axiom_master.xml')
xsd_path = os.path.join(DRIVE_AXIOM, 'axiom_schema.xsd')

if not os.path.exists(xml_path):
    print(f"❌ No encuentro axiom_master.xml")
elif not os.path.exists(xsd_path):
    print(f"❌ No encuentro axiom_schema.xsd")
else:
    try:
        schema_doc = etree.parse(xsd_path)
        schema = etree.XMLSchema(schema_doc)
        xml_doc = etree.parse(xml_path)

        if schema.validate(xml_doc):
            print("✅✅✅ axiom_master.xml VALIDA contra axiom_schema.xsd")
            root = xml_doc.getroot()
            print(f"\n   Schema version:  {root.get('schema_version')}")
            print(f"   Project version: {root.get('project_version')}")
            print(f"   Last modified:   {root.get('last_modified')}")
            print(f"   Seal authority:  {root.get('seal_authority')}")
        else:
            print("❌ XML NO VALIDA contra XSD:")
            for e in schema.error_log:
                print(f"   Línea {e.line}: {e.message}")
    except Exception as e:
        print(f"❌ Error al validar: {e}")

## Paso 9 — Veredicto finalResumen ejecutivo del estado de tu sistema AXIOM. Aquí decides si avanzas o si hay algo que corregir antes.

In [ ]:
print("=" * 70)
print("🔍  AXIOM AUDIT S55 — VEREDICTO FINAL")
print("=" * 70)
print(f"\nFecha de auditoría: {datetime.now().isoformat()}")
print(f"\n📁 Drive:  {len(drive_items)} archivos en SPEL 4.0/AXIOM/")
print(f"📁 GitHub: {len(github_items)} archivos en {GITHUB_PATH}/")

symmetry_ok = not only_in_drive and not only_in_github and not divergent
print(f"\n🔁 Simetría Drive↔GitHub: {'✅ PERFECTA' if symmetry_ok else '⚠️ ASIMÉTRICA'}")
print(f"🔗 Hashchain íntegro:     {'✅ SÍ' if all_ok else '❌ NO'}")

# Re-validar XML para resumen
try:
    schema_ok = schema.validate(xml_doc)
except:
    schema_ok = False
print(f"📋 XML valida vs XSD:     {'✅ SÍ' if schema_ok else '❌ NO'}")

print()
if symmetry_ok and all_ok and schema_ok:
    print("✅✅✅  SISTEMA EN ESTADO CONSISTENTE")
    print("       Puedes avanzar con AXIOM con confianza.")
else:
    print("⚠️  HAY ALGO QUE CORREGIR — Revisa los detalles arriba antes de avanzar.")

print("\n" + "=" * 70)